# vLLM na GPU (Colab T4) — test server pristupa za više korisnika

**Cilj:** Ispitati vLLM kao rešenje za postavljanje LLM-a na jedan jači računar u firmi,
kome ceo tim pristupa istovremeno preko API-ja (OpenAI-kompatibilan format).

**Šta se meri:**
1. Performanse pojedinačnog zahteva (isti prompt kao u Ollama / LM Studio / TextGenWebUI testovima — direktno poređenje)
2. Ponašanje pod opterećenjem: **1, 8 i 16 paralelnih zahteva** (simulacija tima) — ovde vLLM treba da pokaže prednost zahvaljujući *continuous batching*-u

**Modeli (bez HF tokena, staju na T4 16GB):**
| Model | Zašto |
|---|---|
| Qwen2.5-Coder-7B-Instruct (AWQ 4-bit) | Isti model kao u prethodnim testovima → poređenje alata |
| Qwen3-8B (AWQ 4-bit) | Novija generacija, vrh male klase na benchmarcima |
| Qwen2.5-Coder-1.5B-Instruct (FP16) | Mali model za kontrast brzine |

**Važna ograničenja (ide u dokumentaciju):**
- vLLM praktično zahteva NVIDIA GPU; CPU režim postoji ali je znatno sporiji od Ollama/llama.cpp na CPU
- Najnovije verzije vLLM-a (0.15+) **ne rade na T4** (compute capability 7.5 / sm_75) → pinujemo stariju verziju
- T4 ne podržava bfloat16 → obavezno `--dtype half` (float16)
- vLLM koristi originalne HF modele (safetensors), ne GGUF kao Ollama/LM Studio; za 16GB VRAM koristimo AWQ 4-bit kvantizaciju (uporedivo sa Q4 GGUF)

In [ ]:
!nvidia-smi

## 1. Instalacija vLLM-a

Pinovana verzija **0.9.2** — poslednja generacija koja pouzdano radi na T4 (novije verzije pucaju na sm_75).
Instalacija traje **5–10 minuta** (povlači svoj PyTorch). Nije potreban restart runtime-a jer server
pokrećemo kao poseban proces, a iz notebook-a mu pristupamo samo preko HTTP-a.

In [ ]:
%%time
!pip install -q vllm==0.9.2

In [ ]:
# Provera da vLLM vidi GPU
!python -c "import torch, vllm; print('vLLM:', vllm.__version__); print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0))"


## 2. Konfiguracija modela i pomoćne funkcije

Server se pokreće komandom `python -m vllm.entrypoints.openai.api_server` i sluša na portu 8000.
Na T4 su ključni parametri:
- `--dtype half` — T4 nema bfloat16
- `--max-model-len 4096` — ograničava kontekst da KV cache stane u 16GB
- `--gpu-memory-utilization 0.90` — koliko VRAM-a vLLM sme da rezerviše
- `VLLM_USE_V1=0` — koristi stariji (V0) engine koji je stabilan na T4

In [ ]:
import subprocess, time, os, requests, json, statistics
from concurrent.futures import ThreadPoolExecutor

PORT = 8000
BASE = f"http://127.0.0.1:{PORT}"
REZULTATI_FAJL = "/content/rezultati_vllm.txt"

MODELI = [
    {
        "naziv": "Qwen2.5-Coder-7B-Instruct-AWQ",
        "hf_id": "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ",
        "extra_args": ["--quantization", "awq"],
        "extra_payload": {},
    },
    {
        "naziv": "Qwen3-8B-AWQ",
        "hf_id": "Qwen/Qwen3-8B-AWQ",
        "extra_args": ["--quantization", "awq"],
        # Qwen3 podrazumevano ukljucuje "thinking" rezim -> iskljucujemo ga
        # da bi merenja bila uporediva sa ostalim modelima
        "extra_payload": {"chat_template_kwargs": {"enable_thinking": False}},
    },
    {
        "naziv": "Qwen2.5-Coder-1.5B-Instruct",
        "hf_id": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
        "extra_args": [],
        "extra_payload": {},
    },
]

server_proc = None
server_log = None

def pokreni_server(model):
    """Pokrece vLLM server u pozadini i ceka da bude spreman."""
    global server_proc, server_log
    zaustavi_server()
    cmd = [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", model["hf_id"],
        "--dtype", "half",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.90",
        "--port", str(PORT),
        "--disable-log-requests",
    ] + model["extra_args"]
    env = dict(os.environ, VLLM_USE_V1="0")
    server_log = open(f"/content/vllm_{model['naziv']}.log", "w")
    server_proc = subprocess.Popen(cmd, stdout=server_log, stderr=subprocess.STDOUT, env=env)
    print(f"Pokrecem server za {model['naziv']} (prvi put ukljucuje i download modela)...")
    start = time.time()
    while True:
        if server_proc.poll() is not None:
            server_log.flush()
            print(open(server_log.name).read()[-3000:])
            raise RuntimeError("Server se ugasio - vidi log iznad")
        try:
            if requests.get(f"{BASE}/health", timeout=2).status_code == 200:
                break
        except requests.exceptions.RequestException:
            pass
        time.sleep(5)
    print(f"Server spreman za {time.time()-start:.0f}s")

def zaustavi_server():
    global server_proc, server_log
    if server_proc is not None:
        server_proc.terminate()
        try:
            server_proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            server_proc.kill()
        server_proc = None
        time.sleep(5)
    if server_log is not None:
        server_log.close()
        server_log = None

def upisi_rezultat(tekst):
    print(tekst)
    with open(REZULTATI_FAJL, "a") as f:
        f.write(tekst + "\n")

print("Konfiguracija ucitana.")

## 3. Funkcije za merenje

- **Pojedinačni zahtev** (streaming): meri *time to first token* (TTFT) i brzinu generisanja (tokens/s) — iste metrike kao Ollama `--verbose` i LM Studio *Prediction Stats*
- **Paralelni zahtevi**: N različitih promptova istovremeno; meri ukupno vreme, ukupan protok servera (tokens/s) i prosečno/najgore vreme čekanja po korisniku

In [ ]:
GLAVNI_PROMPT = "Napisi Python funkciju koja proverava da li je broj prost, sa objasnjenjem koda."

# Razliciti promptovi za simulaciju tima (da prefix cache ne bi ulepsao rezultate)
TIMSKI_PROMPTOVI = [
    "Napisi Python funkciju koja proverava da li je broj prost, sa objasnjenjem koda.",
    "Napisi Python funkciju koja obrce string bez ugradjenih funkcija.",
    "Objasni razliku izmedju liste i tuple u Pythonu sa primerima.",
    "Napisi SQL upit koji vraca 5 najplacenijih zaposlenih po sektoru.",
    "Napisi Python funkciju za binarnu pretragu sortirane liste.",
    "Objasni sta je REST API i navedi primer u Pythonu sa requests bibliotekom.",
    "Napisi funkciju koja racuna n-ti Fibonacijev broj iterativno.",
    "Objasni razliku izmedju git merge i git rebase.",
    "Napisi Python klasu za jednostavan bankovni racun sa uplatom i isplatom.",
    "Napisi regex koji validira email adresu i objasni ga.",
    "Objasni kako radi quicksort i napisi implementaciju u Pythonu.",
    "Napisi Python skriptu koja cita CSV fajl i racuna prosek kolone.",
    "Objasni razliku izmedju procesa i niti (thread) u operativnim sistemima.",
    "Napisi JavaScript funkciju koja debounce-uje pozive druge funkcije.",
    "Objasni sta je Docker kontejner i po cemu se razlikuje od virtuelne masine.",
    "Napisi Python dekorator koji meri vreme izvrsavanja funkcije.",
]

def pojedinacni_zahtev(model, prompt=GLAVNI_PROMPT, max_tokens=512, prikazi_odgovor=True):
    """Streaming zahtev - meri TTFT i tokens/s."""
    payload = {
        "model": model["hf_id"],
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": 0.8,
        "stream": True,
        "stream_options": {"include_usage": True},
        **model["extra_payload"],
    }
    start = time.time()
    ttft = None
    delovi = []
    usage = {}
    with requests.post(f"{BASE}/v1/chat/completions", json=payload, stream=True) as r:
        r.raise_for_status()
        for linija in r.iter_lines():
            if not linija or not linija.startswith(b"data: "):
                continue
            data = linija[len(b"data: "):]
            if data == b"[DONE]":
                break
            chunk = json.loads(data)
            if chunk.get("usage"):
                usage = chunk["usage"]
            for izbor in chunk.get("choices", []):
                sadrzaj = izbor.get("delta", {}).get("content")
                if sadrzaj:
                    if ttft is None:
                        ttft = time.time() - start
                    delovi.append(sadrzaj)
    ukupno = time.time() - start
    tekst = "".join(delovi)
    comp = usage.get("completion_tokens", 0)
    tps = comp / (ukupno - (ttft or 0)) if comp and ukupno > (ttft or 0) else 0
    if prikazi_odgovor:
        print(tekst)
    return {
        "ttft": ttft, "ukupno": ukupno,
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": comp, "tokens_po_sekundi": tps,
    }

def jedan_blokirajuci(model, prompt, max_tokens=512):
    payload = {
        "model": model["hf_id"],
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens, "temperature": 0.8,
        **model["extra_payload"],
    }
    start = time.time()
    r = requests.post(f"{BASE}/v1/chat/completions", json=payload, timeout=600)
    r.raise_for_status()
    trajanje = time.time() - start
    usage = r.json().get("usage", {})
    return {"trajanje": trajanje, "completion_tokens": usage.get("completion_tokens", 0)}

def paralelni_test(model, n, max_tokens=512):
    """n istovremenih zahteva sa razlicitim promptovima."""
    promptovi = TIMSKI_PROMPTOVI[:n]
    start = time.time()
    with ThreadPoolExecutor(max_workers=n) as ex:
        rezultati = list(ex.map(lambda p: jedan_blokirajuci(model, p, max_tokens), promptovi))
    zid = time.time() - start
    ukupno_tokena = sum(r["completion_tokens"] for r in rezultati)
    latencije = [r["trajanje"] for r in rezultati]
    return {
        "n": n,
        "ukupno_vreme_s": zid,
        "ukupno_gen_tokena": ukupno_tokena,
        "protok_servera_tps": ukupno_tokena / zid if zid else 0,
        "prosecna_latencija_s": statistics.mean(latencije),
        "najgora_latencija_s": max(latencije),
    }

print("Funkcije spremne.")

## 4. Glavni test

Za svaki model: podigni server → 1 pojedinačni zahtev (poređenje sa Ollama/LM Studio) → paralelni testovi sa **1, 8 i 16** korisnika → ugasi server.

> Napomena: prvo pokretanje svakog modela uključuje i download (3–7 GB), pa računaj ~5–10 min po modelu samo za start. Na besplatnom Colabu pazi na trajanje sesije — testove možeš pokretati i model po model.

In [ ]:
SVI_REZULTATI = []

for model in MODELI:
    upisi_rezultat("\n" + "=" * 70)
    upisi_rezultat(f"=== {model['naziv']} (vLLM, Colab T4) ===")
    upisi_rezultat("=" * 70)

    pokreni_server(model)

    # --- Pojedinacni zahtev (isti prompt kao raniji testovi) ---
    upisi_rezultat(f"\nPrompt: {GLAVNI_PROMPT}\n")
    r1 = pojedinacni_zahtev(model)
    upisi_rezultat(f"""
Pojedinacni zahtev:
  Time to First Token: {r1['ttft']:.3f}s
  Tokens/Second:       {r1['tokens_po_sekundi']:.2f}
  Prompt Tokens:       {r1['prompt_tokens']}
  Predicted Tokens:    {r1['completion_tokens']}
  Ukupno vreme:        {r1['ukupno']:.1f}s""")

    # --- Paralelni testovi: 1, 8, 16 korisnika ---
    for n in [1, 8, 16]:
        rp = paralelni_test(model, n)
        SVI_REZULTATI.append({"model": model["naziv"], **rp})
        upisi_rezultat(f"""
Paralelno {n} korisnika:
  Ukupno vreme:        {rp['ukupno_vreme_s']:.1f}s
  Generisano tokena:   {rp['ukupno_gen_tokena']}
  Protok servera:      {rp['protok_servera_tps']:.2f} tokens/s
  Prosecno cekanje:    {rp['prosecna_latencija_s']:.1f}s
  Najgore cekanje:     {rp['najgora_latencija_s']:.1f}s""")

    zaustavi_server()

upisi_rezultat("\nTestovi zavrseni.")

## 5. Zbirna tabela

In [ ]:
import pandas as pd

df = pd.DataFrame(SVI_REZULTATI)
df = df.rename(columns={
    "n": "Paralelnih korisnika",
    "ukupno_vreme_s": "Ukupno vreme (s)",
    "ukupno_gen_tokena": "Gen. tokena",
    "protok_servera_tps": "Protok servera (tok/s)",
    "prosecna_latencija_s": "Prosecno cekanje (s)",
    "najgora_latencija_s": "Najgore cekanje (s)",
})
df = df.round(2)
df

In [ ]:
# Kljucna metrika za dokumentaciju: koliko se protok servera SKALIRA sa brojem korisnika.
# Kod Ollama/LM Studio 16 korisnika bi cekalo u redu (16x sporije);
# vLLM ih obradjuje u batch-u pa ukupan protok raste.
pivot = df.pivot_table(index="model", columns="Paralelnih korisnika",
                       values="Protok servera (tok/s)")
pivot["skaliranje 1->16"] = (pivot[16] / pivot[1]).round(1).astype(str) + "x"
pivot

## 6. Preuzimanje rezultata

Fajl `rezultati_vllm.txt` je u istom formatu kao prethodni rezultati (Ollama CPU, LM Studio) — spreman za objedinjenu dokumentaciju.

In [ ]:
from google.colab import files
files.download(REZULTATI_FAJL)

## Beleške za dokumentaciju (prednosti / mane vLLM pristupa)

**Prednosti:**
- OpenAI-kompatibilan API → svaki alat koji radi sa OpenAI API-jem (IDE dodaci, chat klijenti, skripte) radi i sa vLLM-om samo promenom URL-a
- *Continuous batching* i *PagedAttention* → protok servera raste sa brojem istovremenih korisnika umesto da se zahtevi ređaju u red — ključno za scenario "ceo tim svakodnevno"
- Jedan centralni server = jedan GPU za ceo tim, modeli i podaci ostaju u firmi

**Mane / ograničenja:**
- Praktično zahteva NVIDIA GPU (CPU režim postoji, ali je sporiji od Ollama na CPU); primarno Linux
- Ne koristi GGUF — modeli se povlače sa Hugging Face-a (safetensors, AWQ/GPTQ za kvantizaciju); Llama i Gemma traže HF nalog i prihvatanje licence
- Zahtevnija instalacija i podešavanje od Ollama/LM Studio (verzije, dtype, memorija) — na starijim karticama (T4) mora pinovana verzija i `--dtype half`
- Drži ceo model + KV cache u VRAM-u sve vreme → GPU je rezervisan za tu namenu